In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python

import numpy as np
import pandas as pd
import os
import sys

print("Python environment initialized.")


Python environment initialized.


# QSED HPatches Benchmark — Literal Euclidean-Only Score Matrix

This notebook evaluates the **Quantum-Inspired Local Von Neumann Entropy Feature Descriptor** on the **HPatches** benchmark using the **literal per-component squared-entropy Euclidean transform**:

$$g_{ij, k} = \sqrt{\left| (F^A_{i, k})^2 - (F^B_{j, k})^2 \right|}$$
$$C_{ij} = \|g_{ij}\|_2 = \sqrt{\sum_{k=0}^{D-1} \left| (F^A_{i, k})^2 - (F^B_{j, k})^2 \right|}$$

### Key Properties & Protocol:
- **Raw Von Neumann Entropy Bits**: Descriptors are extracted with `norm="none"` (unnormalized entropy bits).
- **Configurations Evaluated**: $(n, y) \in \{(7, 3), (7, 5), (9, 3), (9, 5)\}$, where descriptor dimension $D = n^2 - 1$:
  - $(n=7, y=3) \implies D = 48$
  - $(n=7, y=5) \implies D = 48$
  - $(n=9, y=3) \implies D = 80$
  - $(n=9, y=5) \implies D = 80$
- **Literal Score Matrix $C$**: Each row $i$ is the Euclidean-value vector for reference keypoint $i$; each column $j$ is the Euclidean-value vector for target keypoint $j$.
- **Mutual Nearest Neighbor (MNN) Matching**: Performed exclusively using the score matrix $C$.
- **Homography Estimation**: Estimated via RANSAC with 3.0px reprojection threshold; corner reprojection error evaluated against ground-truth homographies.
- **HPatches Dataset Only**: No Oxford Affine or MegaDepth benchmarks are downloaded or evaluated.
- **Metrics-Only Retention**: Large pairwise score matrices are computed in memory and discarded after each image pair; only configuration-level reports and per-pair CSVs are persisted.


In [2]:
# Clone or update repository in Kaggle environment, or navigate to repo root
import os, sys

if os.path.exists("/kaggle/working"):
    !git clone https://github.com/yaswanthvuppala/24-D-VonNeumon-Similarity-Q.git /kaggle/working/24-D-VonNeumon-Similarity-Q 2>/dev/null || (cd /kaggle/working/24-D-VonNeumon-Similarity-Q && git pull origin main)
    %cd /kaggle/working/24-D-VonNeumon-Similarity-Q
    !pip install -q -r requirements.txt
else:
    # Ensure current working directory has access to src
    repo_dir = os.path.abspath(".")
    if not os.path.isdir(os.path.join(repo_dir, "src")):
        candidate = os.path.abspath(os.path.join(repo_dir, "24-D-VonNeumon-Similarity-Q"))
        if os.path.isdir(os.path.join(candidate, "src")):
            %cd {candidate}
            repo_dir = candidate
    if repo_dir not in sys.path:
        sys.path.insert(0, repo_dir)
    print(f"Working directory: {os.getcwd()}")


/kaggle/working/24-D-VonNeumon-Similarity-Q


In [3]:
import os

# Discover or download HPatches sequences dataset ONLY (no Oxford, no MegaDepth)
dataset_dir = "data/hpatches-sequences-release"

# Check if pre-mounted in Kaggle input
if not os.path.isdir(dataset_dir) and os.path.isdir("/kaggle/input"):
    for root, dirs, _ in os.walk("/kaggle/input"):
        if "hpatches-sequences-release" in dirs:
            candidate = os.path.join(root, "hpatches-sequences-release")
            if os.path.isdir(candidate) and os.path.isfile(os.path.join(candidate, "v_weapons", "1.ppm")):
                dataset_dir = candidate
                print(f"Found mounted HPatches dataset at: {dataset_dir}")
                break

# Download from official Hugging Face mirror if not found
if not os.path.isdir(dataset_dir):
    os.makedirs("data", exist_ok=True)
    print("Downloading HPatches sequences (~1.28 GB zip)...")
    !wget -c -O data/hpatches-sequences-release.zip "https://huggingface.co/datasets/vbalnt/hpatches/resolve/main/hpatches-sequences-release.zip"
    !unzip -q data/hpatches-sequences-release.zip -d data/
    !rm -f data/hpatches-sequences-release.zip
    dataset_dir = "data/hpatches-sequences-release"

if os.path.isdir(dataset_dir):
    extracted = [d for d in os.listdir(dataset_dir) if os.path.isdir(os.path.join(dataset_dir, d))]
    print(f"HPatches dataset ready: {len(extracted)} sequences found in '{dataset_dir}'.")
else:
    print(f"Dataset directory '{dataset_dir}' not yet populated.")


--2026-10-30 18:39:59--  https://huggingface.co/datasets/vbalnt/hpatches/resolve/main/hpatches-sequences-release.zip
Resolving huggingface.co (huggingface.co)... 13.35.202.97, 13.35.202.40, 13.35.202.34, ...
Connecting to huggingface.co (huggingface.co)|13.35.202.97|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.gcp.cdn.hf.co/xet-bridge-us/67e3e06b388d76021c8bfec0/9df11ca526095d0577e2b6ea0aabd16a981b831880766935812204cc8fe5e051?X-Xet-Cas-Uid=public&xip=aROnW-EMtP4&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27hpatches-sequences-release.zip%3B+filename%3D%22hpatches-sequences-release.zip%22%3B&user_id=public&response-content-type=application%2Fzip&Expires=1793389199&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjdlM2UwNmIzODhkNzYwMjFjOGJmZWMwLzlkZjExY2E1MjYwOTVkMDU3N2UyYjZlYTBhYWJkMTZhOTgxYjgzMTg4MDc2NjkzNTgxMjIwNGNjOGZlNWUwNTFcXD9YLVhldC1DYXMtVWlkPXB1YmxpYyZ4aXA9YVJPblctRU10UDQm

In [4]:
# =============================================================================
# SANITY CELL: LITERAL FORMULA, NEGATIVE RADICANDS, AND MNN RECIPROCITY
# =============================================================================
import numpy as np

print("Running sanity checks for literal Euclidean formula and MNN matching...")

# 1. Verify handling of negative raw radicand ((FA)^2 < (FB)^2)
fa_val = 2.0
fb_val = 3.0
raw_radicand = (fa_val ** 2) - (fb_val ** 2)  # 4.0 - 9.0 = -5.0 < 0
assert raw_radicand < 0, "Test condition requires a negative raw radicand"

# The literal formula takes abs(...) before sqrt, ensuring real positive result
g_val = np.sqrt(np.abs(raw_radicand))
expected_g = np.sqrt(5.0)
assert np.isclose(g_val, expected_g), f"Expected {expected_g}, got {g_val}"
print(f"[Sanity 1/4 PASSED] Negative raw radicand ({raw_radicand}) properly handled by abs: g = {g_val:.4f}")

# 2. Vectorized computation vs explicit scalar formula
np.random.seed(42)
N_A, N_B, D = 15, 20, 48
desc_a_sample = np.random.uniform(0.1, 3.1, (N_A, D))
desc_b_sample = np.random.uniform(0.1, 3.1, (N_B, D))

fa_sq = np.square(desc_a_sample)[:, np.newaxis, :]
fb_sq = np.square(desc_b_sample)[np.newaxis, :, :]
abs_diff = np.abs(fa_sq - fb_sq)
C_vectorized = np.sqrt(np.sum(abs_diff, axis=2))

assert C_vectorized.shape == (N_A, N_B), f"Expected shape {(N_A, N_B)}, got {C_vectorized.shape}"
assert np.all(np.isfinite(C_vectorized)), "Score matrix contains non-finite values!"
assert np.all(C_vectorized >= 0), "Score matrix contains negative values!"

# Verify against explicit scalar formula
for i in range(min(5, N_A)):
    for j in range(min(5, N_B)):
        g_explicit = np.sqrt(np.abs(np.square(desc_a_sample[i]) - np.square(desc_b_sample[j])))
        c_explicit = np.linalg.norm(g_explicit)
        assert np.isclose(C_vectorized[i, j], c_explicit), f"Mismatch at ({i}, {j})"
print(f"[Sanity 2/4 PASSED] Vectorized C matches explicit scalar ||g_ij||_2 across all tested pairs.")

# 3. Exact zero distance on identical descriptors
C_self = np.sqrt(np.sum(np.abs(np.square(desc_a_sample)[:, np.newaxis, :] - np.square(desc_a_sample)[np.newaxis, :, :]), axis=2))
assert np.allclose(np.diag(C_self), 0.0), "Self-distance diagonal must be identically zero!"
print(f"[Sanity 3/4 PASSED] Self-score diagonal is identically 0.0.")

# 4. Verify MNN reciprocity
nn_a_to_b = np.argmin(C_vectorized, axis=1)
nn_b_to_a = np.argmin(C_vectorized, axis=0)
mnn_matches = []
for i in range(N_A):
    j = int(nn_a_to_b[i])
    if nn_b_to_a[j] == i:
        mnn_matches.append((i, j, float(C_vectorized[i, j])))

for i, j, dist in mnn_matches:
    assert np.argmin(C_vectorized[i, :]) == j, f"Row minimum mismatch for match ({i}, {j})"
    assert np.argmin(C_vectorized[:, j]) == i, f"Col minimum mismatch for match ({i}, {j})"
    assert np.isclose(C_vectorized[i, j], dist)
print(f"[Sanity 4/4 PASSED] Found {len(mnn_matches)} mutual nearest neighbors; 100% verified reciprocal!")


Running sanity checks for literal Euclidean formula and MNN matching...
[Sanity 1/4 PASSED] Negative raw radicand (-5.0) properly handled by abs: g = 2.2361
[Sanity 2/4 PASSED] Vectorized C matches explicit scalar ||g_ij||_2 across all tested pairs.
[Sanity 3/4 PASSED] Self-score diagonal is identically 0.0.
[Sanity 4/4 PASSED] Found 9 mutual nearest neighbors; 100% verified reciprocal!


In [5]:
# =============================================================================
# NOTEBOOK-LOCAL EVALUATION PIPELINE: EUCLIDEAN-ONLY
# =============================================================================
import os
import csv
import time
from datetime import datetime
from typing import List, Dict, Tuple, Optional, Any
import numpy as np
from tqdm import tqdm

from src.feature.entropy_descriptor import (
    extract_all_entropy_descriptors,
    detect_default_keypoints,
)
from src.feature.hpatches_adapter import (
    load_hpatches_image,
    load_homography,
    scale_homography,
    estimate_homography_ransac,
    compute_corner_error,
    compute_homography_auc,
    get_hpatches_sequences,
    get_sequence_pairs,
    format_ripe_comparison_table,
)


def compute_literal_score_matrix(desc_a: np.ndarray, desc_b: np.ndarray) -> np.ndarray:
    """
    Computes candidate keypoint-pair score matrix C using the literal formula:
        g_{ij, k} = sqrt(|(F^A_{i, k})^2 - (F^B_{j, k})^2|)
        C_{ij}    = ||g_{ij}||_2 = sqrt(sum_k |(F^A_{i, k})^2 - (F^B_{j, k})^2|)
    """
    n_a = desc_a.shape[0]
    n_b = desc_b.shape[0]
    if n_a == 0 or n_b == 0:
        return np.empty((n_a, n_b), dtype=np.float64)

    fa_sq = np.square(desc_a)[:, np.newaxis, :]  # (N_A, 1, D)
    fb_sq = np.square(desc_b)[np.newaxis, :, :]  # (1, N_B, D)
    abs_diff = np.abs(fa_sq - fb_sq)             # (N_A, N_B, D), handles negative raw radicands
    C = np.sqrt(np.sum(abs_diff, axis=2))        # (N_A, N_B)
    return C


def match_mutual_nn_from_scores(
    score_matrix: np.ndarray,
    max_distance: Optional[float] = None
) -> List[Tuple[int, int, float]]:
    """
    Mutual Nearest Neighbor (MNN) matching based on score matrix C alone.
    Keypoint i in A matches keypoint j in B iff:
        argmin_j C[i, j] == j  AND  argmin_i C[i, j] == i
    """
    n_a, n_b = score_matrix.shape
    if n_a == 0 or n_b == 0:
        return []

    nn_a_to_b = np.argmin(score_matrix, axis=1)
    nn_b_to_a = np.argmin(score_matrix, axis=0)

    matches = []
    for i in range(n_a):
        j = int(nn_a_to_b[i])
        if nn_b_to_a[j] == i:
            dist = float(score_matrix[i, j])
            if max_distance is None or dist <= max_distance:
                matches.append((i, j, dist))
    return matches


def run_hpatches_configuration_euclidean_only(
    dataset_dir: str,
    output_dir: str,
    n: int,
    y: int,
    max_keypoints: int = 1024,
    max_side: int = 480,
    ransac_thresh: float = 3.0
) -> Dict[str, Any]:
    """
    Evaluates one (n, y) configuration across all HPatches sequences using
    the literal Euclidean score matrix for MNN matching.
    """
    os.makedirs(output_dir, exist_ok=True)
    D = n * n - 1

    print("=" * 75)
    print(f"  QSED HPatches Evaluation — Euclidean-Only Formula (n={n}, y={y}, D={D})")
    print("=" * 75)
    print(f"  Normalization      : raw Von Neumann entropy bits (none)")
    print(f"  Matching           : Mutual Nearest Neighbor on literal score matrix C")
    print(f"  Max Keypoints      : {max_keypoints}")
    print(f"  Max Image Side     : {max_side}px")
    print(f"  RANSAC Threshold   : {ransac_thresh}px")
    print(f"  Dataset Directory  : {dataset_dir}")
    print(f"  Output Directory   : {output_dir}")
    print("=" * 75)

    sequences = get_hpatches_sequences(dataset_dir)
    n_illum = sum(1 for s in sequences if s["type"] == "illumination")
    n_view = sum(1 for s in sequences if s["type"] == "viewpoint")
    print(f"  Found {len(sequences)} sequences ({n_illum} illumination, {n_view} viewpoint)")
    print(f"  Total image pairs : {len(sequences) * 5}\n")

    errors = {"illumination": [], "viewpoint": [], "all": []}
    match_counts = {"illumination": [], "viewpoint": [], "all": []}
    failed_homographies = {"illumination": 0, "viewpoint": 0, "all": 0}
    per_pair_records = []

    start_time = time.time()

    for seq_info in tqdm(sequences, desc=f"Eval n={n},y={y}", unit="seq"):
        seq_name = seq_info["name"]
        seq_path = seq_info["path"]
        seq_type = seq_info["type"]

        # 1. Reference image
        ref_path = os.path.join(seq_path, "1.ppm")
        img_ref, scale_ref = load_hpatches_image(ref_path, max_side=max_side)
        kps_ref = detect_default_keypoints(img_ref, max_keypoints=max_keypoints)

        if len(kps_ref) == 0:
            for pair_info in get_sequence_pairs(seq_path):
                per_pair_records.append({
                    "sequence": seq_name, "type": seq_type, "pair": pair_info["pair_name"],
                    "num_kps_ref": 0, "num_kps_tgt": 0, "num_matches": 0, "corner_error": float('inf'),
                })
                errors[seq_type].append(float('inf'))
                errors["all"].append(float('inf'))
                failed_homographies[seq_type] += 1
                failed_homographies["all"] += 1
            continue

        # Extract reference descriptors (raw Von Neumann entropy bits)
        desc_ref = extract_all_entropy_descriptors(
            img_ref, kps_ref, ordering="raster", norm="none", n=n, y=y
        )

        # 2. Evaluate target pairs
        for pair_info in get_sequence_pairs(seq_path):
            img_tgt, scale_tgt = load_hpatches_image(pair_info["tgt_path"], max_side=max_side)
            H_gt = load_homography(pair_info["H_path"])
            H_gt_scaled = scale_homography(H_gt, scale_ref, scale_tgt)

            kps_tgt = detect_default_keypoints(img_tgt, max_keypoints=max_keypoints)
            desc_tgt = extract_all_entropy_descriptors(
                img_tgt, kps_tgt, ordering="raster", norm="none", n=n, y=y
            )

            # Build all-keypoint score matrix C using literal formula
            C = compute_literal_score_matrix(desc_ref, desc_tgt)

            # MNN matching directly on score matrix C
            matches = match_mutual_nn_from_scores(C)
            num_matches = len(matches)

            # Free score matrix immediately (vector retention is metrics-only)
            del C

            # Estimate homography via RANSAC
            H_est = estimate_homography_ransac(
                kps_ref, kps_tgt, matches, ransac_thresh=ransac_thresh
            )

            # Corner reprojection error
            if H_est is not None:
                err = compute_corner_error(H_gt_scaled, H_est, img_ref.shape)
            else:
                err = float('inf')
                failed_homographies[seq_type] += 1
                failed_homographies["all"] += 1

            errors[seq_type].append(err)
            errors["all"].append(err)
            match_counts[seq_type].append(num_matches)
            match_counts["all"].append(num_matches)

            per_pair_records.append({
                "sequence": seq_name,
                "type": seq_type,
                "pair": pair_info["pair_name"],
                "num_kps_ref": len(kps_ref),
                "num_kps_tgt": len(kps_tgt),
                "num_matches": num_matches,
                "corner_error": err,
            })

    elapsed = time.time() - start_time

    # 3. Report generation
    results_text_lines = []
    results_text_lines.append("=" * 75)
    results_text_lines.append(f"  QSED HPatches Benchmark Results (Euclidean-Only Formula)")
    results_text_lines.append("=" * 75)
    results_text_lines.append(f"  Date             : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    results_text_lines.append(f"  Configuration    : n={n}, y={y} (D={D})")
    results_text_lines.append(f"  Normalization    : raw Von Neumann entropy bits (none)")
    results_text_lines.append(f"  Matching         : Mutual Nearest Neighbor on literal score matrix C")
    results_text_lines.append(f"  Max Keypoints    : {max_keypoints}")
    results_text_lines.append(f"  Max Image Side   : {max_side}px")
    results_text_lines.append(f"  RANSAC Threshold : {ransac_thresh}px")
    results_text_lines.append(f"  Total Time       : {elapsed:.1f}s")
    results_text_lines.append("")

    summary_metrics = {}
    for cat in ["illumination", "viewpoint", "all"]:
        cat_errs = errors[cat]
        cat_matches = match_counts[cat]
        cat_failed = failed_homographies[cat]
        if len(cat_errs) == 0:
            continue
        auc = compute_homography_auc(cat_errs)
        summary_metrics[cat] = {"auc": auc, "failed": cat_failed, "total": len(cat_errs)}

        header = f"\n  {cat.upper()} ({len(cat_errs)} pairs)"
        results_text_lines.append(header)
        results_text_lines.append("-" * 55)
        for key, val in auc.items():
            results_text_lines.append(f"    {key:>10s} : {val:6.2f}%")
        if len(cat_matches) > 0:
            mc = np.array(cat_matches)
            results_text_lines.append(
                f"    Matches: mean={np.mean(mc):.1f}, median={np.median(mc):.0f}, min={np.min(mc)}, max={np.max(mc)}"
            )
        results_text_lines.append(f"    Failed homographies: {cat_failed}/{len(cat_errs)}")

    # RIPE Table 2 comparison
    all_auc = summary_metrics.get("all", {}).get("auc", {})
    ripe_table = format_ripe_comparison_table(
        our_auc1=all_auc.get("AUC@1px", 0.0),
        our_auc3=all_auc.get("AUC@3px", 0.0),
        our_auc5=all_auc.get("AUC@5px", 0.0),
        model_name=f"QSED-{D}D-EuclideanOnly (n={n},y={y})",
        dim=D
    )
    results_text_lines.append("\n" + ripe_table + "\n")

    # Save results to output directory
    report_path = os.path.join(output_dir, "hpatches_results.txt")
    with open(report_path, "w", encoding="utf-8") as f:
        f.write("\n".join(results_text_lines))

    csv_path = os.path.join(output_dir, "hpatches_per_pair.csv")
    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=[
            "sequence", "type", "pair", "num_kps_ref", "num_kps_tgt", "num_matches", "corner_error"
        ])
        writer.writeheader()
        for rec in per_pair_records:
            writer.writerow(rec)

    print(f"\n  Results saved to: {report_path}")
    print(f"  Per-pair CSV saved to: {csv_path}\n")
    print("\n".join(results_text_lines))

    return {
        "n": n, "y": y, "D": D,
        "elapsed": elapsed,
        "summary": summary_metrics,
        "ripe_table": ripe_table,
    }


In [6]:
# =============================================================================
# ONE-PAIR SMOKE TEST
# =============================================================================
import os
import cv2
import numpy as np

print("Running one-pair smoke test for Euclidean-only pipeline...")

sample_seq = None
if os.path.isdir(dataset_dir):
    seqs = [d for d in sorted(os.listdir(dataset_dir)) if (d.startswith("i_") or d.startswith("v_")) and os.path.isfile(os.path.join(dataset_dir, d, "1.ppm"))]
    if len(seqs) > 0:
        sample_seq = os.path.join(dataset_dir, seqs[0])

if sample_seq is not None:
    print(f"Using HPatches sequence for smoke test: {os.path.basename(sample_seq)}")
    img_ref, scale_ref = load_hpatches_image(os.path.join(sample_seq, "1.ppm"), max_side=480)
    img_tgt, scale_tgt = load_hpatches_image(os.path.join(sample_seq, "2.ppm"), max_side=480)
    H_gt = load_homography(os.path.join(sample_seq, "H_1_2"))
    H_gt_scaled = scale_homography(H_gt, scale_ref, scale_tgt)
else:
    print("Generating synthetic image pair for smoke test...")
    img_ref = np.zeros((240, 320), dtype=np.float64)
    cv2.rectangle(img_ref, (40, 40), (120, 120), 200, -1)
    cv2.circle(img_ref, (200, 150), 30, 150, -1)
    cv2.putText(img_ref, "QSED", (60, 200), cv2.FONT_HERSHEY_SIMPLEX, 1, 255, 2)
    noise = np.random.RandomState(42).randint(0, 30, img_ref.shape)
    img_ref = np.clip(img_ref + noise, 0, 255)
    H_gt_scaled = np.array([[1.02, 0.02, 4.0], [-0.02, 0.98, -2.0], [0.0, 0.0, 1.0]], dtype=np.float64)
    img_tgt = cv2.warpPerspective(img_ref, H_gt_scaled, (320, 240))

# Test with (n=7, y=3, D=48)
kps_ref = detect_default_keypoints(img_ref, max_keypoints=500)
kps_tgt = detect_default_keypoints(img_tgt, max_keypoints=500)
desc_ref = extract_all_entropy_descriptors(img_ref, kps_ref, norm="none", n=7, y=3)
desc_tgt = extract_all_entropy_descriptors(img_tgt, kps_tgt, norm="none", n=7, y=3)

C = compute_literal_score_matrix(desc_ref, desc_tgt)
assert C.shape == (len(kps_ref), len(kps_tgt)), f"Shape mismatch: {C.shape}"
assert np.all(np.isfinite(C)), "C contains non-finite values"

matches = match_mutual_nn_from_scores(C)
# Verify reciprocity on every match
for i, j, dist in matches:
    assert np.argmin(C[i, :]) == j
    assert np.argmin(C[:, j]) == i
del C  # Discard large score matrix

H_est = estimate_homography_ransac(kps_ref, kps_tgt, matches, ransac_thresh=3.0)
assert H_est is not None, "RANSAC failed to estimate homography!"
corner_err = compute_corner_error(H_gt_scaled, H_est, img_ref.shape)

print(f"Smoke test successful!")
print(f"  Keypoints ref={len(kps_ref)}, tgt={len(kps_tgt)}")
print(f"  Reciprocal MNN matches: {len(matches)}")
print(f"  Corner reprojection error: {corner_err:.4f}px")


Running one-pair smoke test for Euclidean-only pipeline...
Using HPatches sequence for smoke test: i_ajuntament
Smoke test successful!
  Keypoints ref=500, tgt=500
  Reciprocal MNN matches: 166
  Corner reprojection error: 0.4243px


In [7]:
# =============================================================================
# FULL HPATCHES BENCHMARK: EVALUATE ALL (n, y) CONFIGURATIONS
# =============================================================================
window_configs = [(7, 3), (7, 5), (9, 3), (9, 5)]
results_by_config = {}

for n, y in window_configs:
    d = n * n - 1
    output_dir = f"outputs/hpatches_euclidean_only_n{n}_y{y}"
    print(f"\n{'='*75}\nExecuting Configuration: n={n}, y={y} (D={d})\n{'='*75}")
    res = run_hpatches_configuration_euclidean_only(
        dataset_dir=dataset_dir,
        output_dir=output_dir,
        n=n, y=y,
        max_keypoints=1024,
        max_side=480,
        ransac_thresh=3.0
    )
    results_by_config[(n, y)] = res



Executing Configuration: n=7, y=3 (D=48)
  QSED HPatches Evaluation — Euclidean-Only Formula (n=7, y=3, D=48)
  Normalization      : raw Von Neumann entropy bits (none)
  Matching           : Mutual Nearest Neighbor on literal score matrix C
  Max Keypoints      : 1024
  Max Image Side     : 480px
  RANSAC Threshold   : 3.0px
  Dataset Directory  : data/hpatches-sequences-release
  Output Directory   : outputs/hpatches_euclidean_only_n7_y3
  Found 116 sequences (57 illumination, 59 viewpoint)
  Total image pairs : 580



Eval n=7,y=3: 100%|██████████| 116/116 [25:16<00:00, 13.07s/seq]



  Results saved to: outputs/hpatches_euclidean_only_n7_y3/hpatches_results.txt
  Per-pair CSV saved to: outputs/hpatches_euclidean_only_n7_y3/hpatches_per_pair.csv

  QSED HPatches Benchmark Results (Euclidean-Only Formula)
  Date             : 2026-10-30 19:05:56
  Configuration    : n=7, y=3 (D=48)
  Normalization    : raw Von Neumann entropy bits (none)
  Matching         : Mutual Nearest Neighbor on literal score matrix C
  Max Keypoints    : 1024
  Max Image Side   : 480px
  RANSAC Threshold : 3.0px
  Total Time       : 1516.0s


  ILLUMINATION (285 pairs)
-------------------------------------------------------
       AUC@1px :  36.53%
       Acc@1px :  57.19%
       AUC@3px :  57.52%
       Acc@3px :  72.63%
       AUC@5px :  63.66%
       Acc@5px :  73.33%
      AUC@10px :  69.14%
      Acc@10px :  75.09%
    Matches: mean=284.5, median=247, min=57, max=813
    Failed homographies: 0/285

  VIEWPOINT (295 pairs)
-------------------------------------------------------
       AUC

Eval n=7,y=5: 100%|██████████| 116/116 [31:36<00:00, 16.35s/seq]



  Results saved to: outputs/hpatches_euclidean_only_n7_y5/hpatches_results.txt
  Per-pair CSV saved to: outputs/hpatches_euclidean_only_n7_y5/hpatches_per_pair.csv

  QSED HPatches Benchmark Results (Euclidean-Only Formula)
  Date             : 2026-10-30 19:37:32
  Configuration    : n=7, y=5 (D=48)
  Normalization    : raw Von Neumann entropy bits (none)
  Matching         : Mutual Nearest Neighbor on literal score matrix C
  Max Keypoints    : 1024
  Max Image Side   : 480px
  RANSAC Threshold : 3.0px
  Total Time       : 1896.3s


  ILLUMINATION (285 pairs)
-------------------------------------------------------
       AUC@1px :  28.67%
       Acc@1px :  46.67%
       AUC@3px :  44.81%
       Acc@3px :  56.49%
       AUC@5px :  49.99%
       Acc@5px :  58.25%
      AUC@10px :  55.16%
      Acc@10px :  61.05%
    Matches: mean=340.2, median=325, min=82, max=821
    Failed homographies: 0/285

  VIEWPOINT (295 pairs)
-------------------------------------------------------
       AUC

Eval n=9,y=3: 100%|██████████| 116/116 [38:38<00:00, 19.99s/seq]



  Results saved to: outputs/hpatches_euclidean_only_n9_y3/hpatches_results.txt
  Per-pair CSV saved to: outputs/hpatches_euclidean_only_n9_y3/hpatches_per_pair.csv

  QSED HPatches Benchmark Results (Euclidean-Only Formula)
  Date             : 2026-10-30 20:16:11
  Configuration    : n=9, y=3 (D=80)
  Normalization    : raw Von Neumann entropy bits (none)
  Matching         : Mutual Nearest Neighbor on literal score matrix C
  Max Keypoints    : 1024
  Max Image Side   : 480px
  RANSAC Threshold : 3.0px
  Total Time       : 2318.6s


  ILLUMINATION (285 pairs)
-------------------------------------------------------
       AUC@1px :  38.18%
       Acc@1px :  61.40%
       AUC@3px :  62.89%
       Acc@3px :  81.75%
       AUC@5px :  70.98%
       Acc@5px :  84.21%
      AUC@10px :  78.11%
      Acc@10px :  86.32%
    Matches: mean=260.9, median=220, min=33, max=818
    Failed homographies: 0/285

  VIEWPOINT (295 pairs)
-------------------------------------------------------
       AUC

Eval n=9,y=5: 100%|██████████| 116/116 [47:55<00:00, 24.79s/seq]


  Results saved to: outputs/hpatches_euclidean_only_n9_y5/hpatches_results.txt
  Per-pair CSV saved to: outputs/hpatches_euclidean_only_n9_y5/hpatches_per_pair.csv

  QSED HPatches Benchmark Results (Euclidean-Only Formula)
  Date             : 2026-10-30 21:04:06
  Configuration    : n=9, y=5 (D=80)
  Normalization    : raw Von Neumann entropy bits (none)
  Matching         : Mutual Nearest Neighbor on literal score matrix C
  Max Keypoints    : 1024
  Max Image Side   : 480px
  RANSAC Threshold : 3.0px
  Total Time       : 2875.3s


  ILLUMINATION (285 pairs)
-------------------------------------------------------
       AUC@1px :  30.92%
       Acc@1px :  50.88%
       AUC@3px :  51.44%
       Acc@3px :  65.96%
       AUC@5px :  58.23%
       Acc@5px :  69.47%
      AUC@10px :  64.45%
      Acc@10px :  71.58%
    Matches: mean=320.5, median=296, min=59, max=833
    Failed homographies: 0/285

  VIEWPOINT (295 pairs)
-------------------------------------------------------
       AUC

In [8]:
# =============================================================================
# CONSOLIDATED SUMMARY: ALL CONFIGURATIONS (EUCLIDEAN-ONLY)
# =============================================================================
print("=" * 80)
print("   QSED LITERAL EUCLIDEAN-ONLY: CONSOLIDATED RESULTS ACROSS CONFIGURATIONS")
print("=" * 80)
print(f"{'Config (n, y)':<15} | {'Dim':<5} | {'AUC@1px':<10} | {'AUC@3px':<10} | {'AUC@5px':<10} | {'Failed H':<10} | {'Time':<8}")
print("-" * 80)

for (n, y), res in results_by_config.items():
    d = n * n - 1
    all_metrics = res["summary"].get("all", {})
    auc = all_metrics.get("auc", {})
    failed = all_metrics.get("failed", 0)
    total = all_metrics.get("total", 0)
    elapsed = res["elapsed"]
    print(f"n={n}, y={y:<8} | {d:<5} | {auc.get('AUC@1px', 0.0):>7.2f}%  | {auc.get('AUC@3px', 0.0):>7.2f}%  | {auc.get('AUC@5px', 0.0):>7.2f}%  | {failed}/{total:<5} | {elapsed:>6.1f}s")
print("=" * 80)
print("Benchmark complete. HPatches was the only dataset evaluated (no Oxford, no MegaDepth).")


   QSED LITERAL EUCLIDEAN-ONLY: CONSOLIDATED RESULTS ACROSS CONFIGURATIONS
Config (n, y)   | Dim   | AUC@1px    | AUC@3px    | AUC@5px    | Failed H   | Time    
--------------------------------------------------------------------------------
n=7, y=3        | 48    |   23.31%  |   43.05%  |   50.36%  | 0/580   | 1516.0s
n=7, y=5        | 48    |   18.21%  |   34.19%  |   40.24%  | 0/580   | 1896.3s
n=9, y=3        | 80    |   23.74%  |   46.56%  |   55.30%  | 0/580   | 2318.6s
n=9, y=5        | 80    |   19.88%  |   40.05%  |   47.91%  | 0/580   | 2875.3s
Benchmark complete. HPatches was the only dataset evaluated (no Oxford, no MegaDepth).
